In [26]:
import zipfile
import numpy as np
import pandas as pd

import networkx as nx

from code_utils.utils_basic import PROJECTED_CRS, PROJECT_PATH

from code_utils.topological_metrics.g_clustering_coefficient import clustering_coefficient
from code_utils.topological_metrics.g_katz_centrality import select_katz_centrality_alpha
from code_utils.topological_metrics.g_degree_strength_centrality import degree_centrality, strength

In [27]:
zip_path = PROJECT_PATH / 'data/bus_network/bus_graph_20221020.zip'
with zipfile.ZipFile(zip_path, 'r') as zf:
    # Load L space graph
    with zf.open('graph_space_l.graphml') as f:
        G_l = nx.read_graphml(f)
    # Load P space graph
    with zf.open('graph_space_p.graphml') as f:
        G_p = nx.read_graphml(f)

#### Clustering coefficient

In [28]:
for G in [G_l, G_p]:
    for weight in ['distance', None]:
        cc_1 = clustering_coefficient(G, weight=weight, normalized_mat=True)
        cc_1 = pd.Series(cc_1)

        cc_2 = nx.clustering(G, weight=weight)
        cc_2 = pd.Series(cc_2)

        error = cc_1.sub(cc_2).abs().sum()
        print(error)

1.7984203536103305e-16
0.0
6.711905337075663e-14
0.0


In [ ]:
def kate_centrality(graph, alpha, weight=None):

    node_list = list(graph.nodes())

    node_num = len(node_list)

    adj_mat = nx.adjacency_matrix(graph, weight=weight).toarray()

    # check "alpha"
    eigenvalues = np.linalg.eigvals(adj_mat)
    max_eigenvalue = np.max(eigenvalues.real)

    try:
        assert alpha <= 1 / max_eigenvalue
    except:
        print('alpha must be no more than the reciprocal of the maximum eigenvalue of adjacency matrix,' \
              'but get alpha = {0}, and 1 / max eigenvalue = {1}'.format(alpha, 1 / max_eigenvalue))
        sys.exit(0)

    inv = np.linalg.inv(np.eye(node_num) - alpha * adj_mat.T)
    kc = np.dot(inv, np.ones(node_num)) - np.ones(node_num)

    kc = {k : v for k, v in zip(node_list, kc.tolist())}

    return kc
# ============================================================
def nx_kate_centrality(graph, alpha, weight=None):

    kc = nx.katz_centrality(graph, alpha=alpha,
                            beta=1.0,
                            weight=weight,
                            max_iter=100, tol=1.0e-9,
                            normalized=False)

    # minus one
    kc = {k : v - 1. for k, v in kc.items()}

    return kc
# ============================================================

In [16]:
select_katz_centrality_alpha(G_p, weight=None)

(0.001, 129.49592279272898)

In [25]:
degree_centrality(G_p, normalized=True)

({'01012': 0.05281828931809223,
  '01112': 0.036657469452108786,
  '01113': 0.03764288529759558,
  '01121': 0.061292865589278676,
  '01211': 0.06976744186046512,
  '01311': 0.06996452502956248,
  '01549': 0.00650374458021285,
  '01559': 0.009854158454867954,
  '07371': 0.0607016160819866,
  '60011': 0.04375246353961371,
  '60021': 0.04040204966495861,
  '60031': 0.04059913283405597,
  '60159': 0.06681119432400473,
  '60161': 0.07232952305873078,
  '60211': 0.04414662987780843,
  '60229': 0.03310997240835632,
  '62011': 0.05400078833267639,
  '62021': 0.05419787150177375,
  '62031': 0.054394954670871104,
  '62041': 0.054592037839968466,
  '62051': 0.05932203389830508,
  '62239': 0.035277887268427274,
  '62249': 0.03508080409932991,
  '63011': 0.039219550650374455,
  '63021': 0.039416633819471816,
  '63031': 0.04513204572329523,
  '63041': 0.04887662593614505,
  '63051': 0.04907370910524241,
  '63061': 0.05951911706740244,
  '63079': 0.03508080409932991,
  '63089': 0.03488372093023256,
 